# 03 Train Multiple Models


In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path('..')/'src'))


## Objectives

- Train **multiple supervised models** to predict course success.
- Compare models using a consistent split.
- Select the best model and persist it as an artifact for the Streamlit app.

**Targets:**
- Primary: `log_profit = log1p(profit)` (handles heavy-tail)

**Output:** `models/profit_model.joblib`


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

train_df = pd.read_parquet(Path('..')/'data'/'processed'/'train.parquet')
test_df  = pd.read_parquet(Path('..')/'data'/'processed'/'test.parquet')
train_df.shape, test_df.shape


((2941, 21), (736, 21))

In [3]:
from course_rec.models import train_multiple_regression_models

# NOTE: the helper trains on structured features and target='profit'.
# For log target, we train manually below (recommended).


### A) Structured-feature models on `log_profit` (recommended)

We train multiple models on `log_profit`, then report metrics on `log_profit`.


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

numeric_cols = ['price','num_subscribers','num_reviews','num_lectures','content_duration_hours','year','month','day','title_len']
categorical_cols = ['subject','level','is_paid']

X_train = train_df[numeric_cols + categorical_cols]
y_train = train_df['log_profit']
X_test  = test_df[numeric_cols + categorical_cols]
y_test  = test_df['log_profit']

num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                    ('ohe', OneHotEncoder(handle_unknown='ignore'))])

pre = ColumnTransformer([
    ('num', num_pipe, numeric_cols),
    ('cat', cat_pipe, categorical_cols)
], remainder='drop')

candidates = {
    'ridge': Ridge(alpha=5.0, random_state=42),
    'random_forest': RandomForestRegressor(n_estimators=400, min_samples_leaf=2, n_jobs=-1, random_state=42),
    'hist_gbdt': HistGradientBoostingRegressor(max_iter=500, learning_rate=0.06, random_state=42),
}

rows=[]
fit_pipes={}
for name, model in candidates.items():
    pipe = Pipeline([('pre', pre), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    mae  = float(mean_absolute_error(y_test, pred))
    r2   = float(r2_score(y_test, pred))

    rows.append({'model': name, 'rmse_log': rmse, 'mae_log': mae, 'r2_log': r2})
    fit_pipes[name]=pipe

pd.DataFrame(rows).sort_values('rmse_log')


,model,rmse_log,mae_log,r2_log
2,hist_gbdt,0.076406,0.038083,0.999610
1,random_forest,0.079404,0.035621,0.999579
0,ridge,2.076478,1.568122,0.712196


In [5]:
# Pick best (lowest RMSE on log scale)
leaderboard = pd.DataFrame(rows).sort_values('rmse_log')
best_name = leaderboard.iloc[0]['model']
best_pipe = fit_pipes[best_name]
(best_name, leaderboard.iloc[0].to_dict())


('hist_gbdt',
 {'model': 'hist_gbdt',
  'rmse_log': 0.0764056781049383,
  'mae_log': 0.03808292449589351,
  'r2_log': 0.9996103334712475})

In [6]:
# Optional: interpret in original profit units for sanity
pred_log = best_pipe.predict(X_test)
pred_profit = np.expm1(pred_log)
true_profit = np.expm1(y_test)

rmse_profit = float(np.sqrt(mean_squared_error(true_profit, pred_profit)))
mae_profit  = float(mean_absolute_error(true_profit, pred_profit))
(rmse_profit, mae_profit)


(228071.07647099294, 29611.647529709608)

In [7]:
# Save model artifact
import joblib

models_dir = Path('..')/'models'
models_dir.mkdir(parents=True, exist_ok=True)
MODEL_PATH = models_dir/'profit_model.joblib'
joblib.dump({'model_name': best_name, 'pipeline': best_pipe, 'numeric_cols': numeric_cols, 'categorical_cols': categorical_cols}, MODEL_PATH)
MODEL_PATH


WindowsPath('../models/profit_model.joblib')